<a href="https://colab.research.google.com/github/luca-fce/siniestros-futbol-caba/blob/main/notebook/3_1_seleccion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np

###Carga y exploración inicial

La base de resultados internacionales ([Kaggle](https://www.kaggle.com/datasets/martj42/international-football-results-from-1872-to-2017)) contiene el historial global de todos los partidos entre 1872 y 2026. Para optimizar el procesamiento y acotar el alcance temporal al del trabajo práctico (2021 - 2025), aplicamos dos filtros:

Filtro temporal (2021-2025).

Filtro de variable (Partidos de la Selección Argentina).

In [1]:
# 1. Definición de la fuente
url_futbol = 'https://raw.githubusercontent.com/luca-fce/siniestros-futbol-caba/refs/heads/main/datos/Resultados%20Internacionales%201872%20-%202026/results.csv'

# 2. Carga y tipado temporal
df_futbol = pd.read_csv(url_futbol, parse_dates=['date'])

# 3. Creación de máscaras de filtrado
filtro_fechas = (df_futbol['date'] >= '2021-01-01') & (df_futbol['date'] <= '2025-12-31')
filtro_argentina = (df_futbol['home_team'] == 'Argentina') | (df_futbol['away_team'] == 'Argentina')

# 4. Aplicación y recorte de memoria
df_argentina = df_futbol[filtro_fechas & filtro_argentina].copy()

# 5. Validación
print(f"Base reducida a {df_argentina.shape[0]} partidos oficiales y amistosos.")
display(df_argentina.head())

Base reducida a 67 partidos oficiales y amistosos.


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
44015,2021-06-03,Argentina,Chile,1,1,FIFA World Cup qualification,Santiago del Estero,Argentina,False
44110,2021-06-08,Colombia,Argentina,2,2,FIFA World Cup qualification,Barranquilla,Colombia,False
44161,2021-06-14,Argentina,Chile,1,1,Copa América,Rio de Janeiro,Brazil,True
44198,2021-06-18,Argentina,Uruguay,1,0,Copa América,Brasília,Brazil,True
44213,2021-06-21,Argentina,Paraguay,1,0,Copa América,Brasília,Brazil,True


###Transformación de variables
En la base original falta en los partidos la variable horaria. Para responder nuestra pregunta de investigación con precisión, realizamos las siguientes transformaciones:
1. **Asignación de horarios:** Mapeamos manualmente los horarios atípicos (ej. Mundial de Qatar) y asignamos las 20:00 hs como horario estándar por defecto para el resto de los encuentros.
2. **Creación de variables datetime:** Generamos la columna `inicio_ventana_12h` (estimando el pitazo final 2 horas después del inicio) y `fin_ventana_12h` (sumando 12 horas al pitazo final). Esta franja será el "embudo" para filtrar los siniestros viales.

In [ ]:
import pandas as pd

# 1. Convertimos la columna de fecha a texto (string) para que sea fácil cruzarla con nuestro diccionario
df_argentina['fecha_str'] = df_argentina['date'].dt.strftime('%Y-%m-%d')

# 2. Diccionario de horarios atípicos (Mundial Qatar 2022 que se jugó a la mañana/mediodía)
# Si encuentran otros partidos en horarios raros, los agregan acá.
horarios_atipicos = {
    '2022-11-22': '07:00:00', # Arabia Saudita
    '2022-11-26': '16:00:00', # Mexico
    '2022-11-30': '16:00:00', # Polonia
    '2022-12-03': '16:00:00', # Australia
    '2022-12-09': '16:00:00', # Paises Bajos
    '2022-12-13': '16:00:00', # Croacia
    '2022-12-18': '12:00:00'  # Final Francia
}

# 3. Mapeo: busca la fecha en el diccionario, si la encuentra pone esa hora, sino (fillna) clava '20:00:00'
df_argentina['hora_partido'] = df_argentina['fecha_str'].map(horarios_atipicos).fillna('20:00:00')

# 4. Magia de Pandas: Juntamos la fecha y la hora en una sola columna de tipo "datetime" real
df_argentina['datetime_inicio'] = pd.to_datetime(df_argentina['fecha_str'] + ' ' + df_argentina['hora_partido'])

# 5. Transformaciones temporales exigidas por la cátedra
# El riesgo de chocar por enojo empieza cuando termina el partido (aprox 2 horas después del inicio)
df_argentina['inicio_ventana'] = df_argentina['datetime_inicio'] + pd.Timedelta(hours=2)

# El riesgo termina 12 horas después
df_argentina['fin_ventana'] = df_argentina['inicio_ventana'] + pd.Timedelta(hours=12)

# 6. Borramos columnas intermedias que ya no sirven para mantener limpio el dataframe
df_argentina = df_argentina.drop(columns=['fecha_str', 'hora_partido', 'datetime_inicio'])

# Mostramos el resultado de la transformación
print("Ventanas de 12 horas creadas con éxito.")
display(df_argentina[['date', 'opponent', 'inicio_ventana', 'fin_ventana']].head(7))